# Yahoo Finance case study

This notebook demonstrates the **data-to-package boundary** with a convenient public downloader. It is an educational example, not a reproducible data fixture or investment backtest.

Important boundaries:

- `yfinance` is not part of MFDRO and is not affiliated with Yahoo.
- The downloader requires network access and its response can change. Preserve a licensed source snapshot if exact reconstruction matters.
- Review Yahoo's terms; the yfinance project states that Yahoo Finance data are intended for personal use only.
- The fixed ticker list below is not a point-in-time universe and may contain survivorship or selection bias. It must not be interpreted as backtest evidence.

References: [yfinance download API](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html), [yfinance project notice](https://github.com/ranaroussi/yfinance).

## Install optional dependencies

From the package root, install the notebook extra before starting Jupyter:

```bash
python -m pip install -e ".[notebooks]"
```

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

import mfdro
from mfdro import MultiFrequencySignal, SignalConfig

plt.style.use("seaborn-v0_8-whitegrid")
INK = "#183b4e"
ACCENT = "#008c82"
print(f"MFDRO {mfdro.__version__}; yfinance {yf.__version__}")

## 1. Download explicitly configured adjusted prices

The start date is inclusive and the end date is exclusive in the documented `download` API. `auto_adjust=True` is passed explicitly rather than relying on a library default.

In [ ]:
TICKERS = ["SPY", "QQQ", "IWM", "EFA", "AGG"]
START = "2015-01-01"
END = "2025-01-01"  # exclusive

raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    auto_adjust=True,
    actions=False,
    progress=False,
    group_by="column",
    multi_level_index=True,
    threads=True,
)

if raw.empty:
    raise RuntimeError(
        "Yahoo Finance returned no data; retry later and inspect the network response."
    )

raw.tail()

## 2. Make the price-to-return policy visible

With `auto_adjust=True`, this example takes the returned adjusted `Close` field. It measures coverage before enforcing a balanced panel, preserves requested ticker order, and calculates simple returns with no fill. Dropping incomplete rows is an explicit convenience policy—not a universally valid production cleaning rule.

In [ ]:
close = raw["Close"].copy()
if isinstance(close, pd.Series):
    close = close.to_frame(name=TICKERS[0])
close = close.reindex(columns=TICKERS).sort_index()

coverage = close.notna().mean().rename("price_coverage")
prices = close.dropna(axis=0, how="any")
returns = prices.pct_change(fill_method=None).dropna(axis=0, how="any")
returns.index.name = "date"

if returns.empty:
    raise RuntimeError("The explicit cleaning policy left no complete return observations.")

coverage, returns.describe().T

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.7))
(prices / prices.iloc[0]).plot(ax=axes[0], linewidth=1.1)
axes[0].set(title="Normalized adjusted-close series", xlabel="date", ylabel="growth of 1")
coverage.plot.bar(ax=axes[1], color=INK, rot=0)
axes[1].set(
    title="Vendor response coverage before row filtering",
    xlabel="ticker",
    ylabel="observed fraction",
    ylim=(0, 1.05),
)
figure.tight_layout()

For research-grade reconstruction, save the raw vendor response plus request parameters, retrieval timestamp, package versions, licensing context, and checksums. A future call with the same dates is not an immutable source.

## 3. Preflight and estimate

The example does not invent an authoritative exchange calendar from the downloaded index. Supply a genuine point-in-time calendar in production.

In [ ]:
config = SignalConfig.projected(
    n_projections=100,
    n_quantiles=100,
    random_state=20250301,
)
engine = MultiFrequencySignal(config)

diagnostics = engine.validate_path_inputs(
    returns,
    lookback_months=36,
    seed_namespace="yfinance_example",
)
diagnostics.summary()

In [ ]:
if not diagnostics.is_usable:
    raise RuntimeError("No formation has enough data; inspect diagnostics.formations.")

path = engine.estimate_path(
    returns,
    lookback_months=36,
    on_insufficient="skip",
    seed_namespace="yfinance_example",
)

path.summary(), path.estimates[["date", "rho", "sqrt_rho"]].tail()

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(10, 5.8), sharex=True)
axes[0].plot(path.sqrt_rho.index, path.sqrt_rho, color=INK, linewidth=1.7)
axes[0].set(title="Illustrative cross-frequency dispersion", ylabel=r"$\sqrt{\rho}$")
for column, color in zip(
    ["n_daily", "n_weekly", "n_monthly"], [INK, ACCENT, "#d17a22"], strict=True
):
    axes[1].plot(
        path.audit["date"], path.audit[column], label=column.removeprefix("n_"), color=color
    )
axes[1].set(xlabel="formation date", ylabel="observations per measure")
axes[1].legend(frameon=False, ncol=3)
figure.tight_layout()

## What this result does—and does not—show

The path shows how the package behaves on one explicitly transformed download. It does not validate the vendor data, establish point-in-time ETF eligibility, predict returns, calibrate a DRO radius, or demonstrate a profitable strategy. Replace the convenience download with a governed data pipeline before drawing research conclusions.